# Dataset 03 | MusicBrainz API

## 1. Objetivo

Evaluar la API de MusicBrainz como posible fuente de información musical para el proyecto **Music Recommender | SQL Edition**.

El objetivo es determinar si permite complementar los datasets analizados anteriormente (Spotify Tracks 114K y FMA) mediante información sobre artistas, grabaciones, lanzamientos y relaciones musicales.

## 2. Descripción de la fuente

MusicBrainz es una base de datos musical abierta que proporciona información estructurada sobre artistas, grabaciones, lanzamientos y otras entidades musicales.

A diferencia de los archivos CSV anteriores, MusicBrainz permite consultar información mediante una API.

## 3. Criterios de evaluación

Se analizarán los siguientes aspectos:

1. **Acceso a los datos:** comprobar cómo realizar consultas a la API desde Python.
2. **Información disponible:** identificar los campos relacionados con artistas, canciones y álbumes.
3. **Identificadores únicos:** comprobar cómo identifica MusicBrainz sus registros.
4. **Calidad de los datos:** revisar campos disponibles, valores ausentes y posibles limitaciones.
5. **Relaciones musicales:** evaluar las conexiones entre artistas, grabaciones y lanzamientos.
6. **Integración con SQL:** determinar cómo podríamos almacenar y relacionar la información obtenida.
7. **Utilidad para recomendaciones:** valorar si aporta información que complemente Spotify 114K y FMA.

## 4. Metodología

La exploración se realizará con Python mediante consultas de prueba a la API.

Se trabajará inicialmente con un número reducido de registros para comprender la estructura de las respuestas.

No se pretende descargar el catálogo completo, sino evaluar la utilidad de MusicBrainz para el proyecto.

## 5. Resultado esperado

Al finalizar la exploración se elaborará un resumen con:

- Datos obtenidos.
- Estructura de las respuestas.
- Posibilidades de integración con SQL.
- Ventajas y limitaciones.
- Valoración de MusicBrainz como fuente principal o complementaria.

**Estado:** pendiente de exploración.

In [9]:
import requests
import pandas as pd

In [10]:
url = "https://musicbrainz.org/ws/2/artist/"

params = {
    "query": 'artist:"BTS"',
    "fmt": "json",
    "limit": 5
}

headers = {
    "User-Agent": "MusicRecommenderSQL/1.0 (educational project)"
}

response = requests.get(
    url,
    params=params,
    headers=headers,
    timeout=15
)

print("Código de respuesta:", response.status_code)

if response.status_code == 200:
    data = response.json()
    display(pd.DataFrame(data["artists"]).head())
else:
    print(response.text[:300])

Código de respuesta: 503
{"error": "The MusicBrainz web server is currently busy. Please try again later."}



In [11]:
artists_df = pd.DataFrame(data["artists"])

display(
    artists_df[
        ["id", "name", "type", "country", "disambiguation", "score"]
    ]
)

,id,name,type,country,disambiguation,score
0,0d79fe8e-ba27-4859-bb8c-2f255f346853,BTS,Group,KR,South Korean boy group,100
1,57729a9a-0238-414e-a3d4-b16b26c4a49d,BTS,Person,NaN,"US rapper, Born To Spit",73
2,3ec446d4-0437-4f0a-8060-6c13e5e77a84,BTS,Group,IE,Irish New Wave band,72
3,f06bf088-0ab6-4aa1-911b-b87a705499f2,BTS,NaN,NaN,jazz/funk/soul,72
4,0c1d9174-709d-43ab-bc95-733ef61abe2a,Big Time Sarah and the BTS Express,Group,US,NaN,55


In [13]:
bts_id = artists_df.loc[0, "id"]

url = "https://musicbrainz.org/ws/2/recording/"

params = {
    "query": f"arid:{bts_id}",
    "fmt": "json",
    "limit": 10
}

response = requests.get(
    url,
    params=params,
    headers=headers,
    timeout=15
)

print("Código de respuesta:", response.status_code)

if response.status_code == 200:
    recordings_data = response.json()

    recordings_df = pd.DataFrame(
        recordings_data["recordings"]
    )

    display(recordings_df.head())
else:
    print(response.text[:300])

Código de respuesta: 200


,id,score,artist-credit-id,title,length,video,artist-credit,first-release-date,releases,isrcs,disambiguation
0,0ba51e80-b418-4402-9ab3-0cdd4a024873,100,958b3f14-0897-34a3-aed7-95cc918c7c23,A Brand New Day,205000,None,"[{'joinphrase': ' & ', 'name': 'BTS', 'artist'...",2019-06-14,[{'id': '5511b5fd-ca9a-4371-93d7-4a07612458bf'...,[QM6N21956069],NaN
1,5793a212-fa86-45a3-9b10-32edbb2db2bb,100,514b698a-5f5a-49be-a0c2-f970926ac198,A Song Make to You Smile,189173,None,"[{'joinphrase': ' feat. ', 'name': 'Lee Seung ...",2020-12-10,[{'id': '905a4e01-1a11-470a-b8c4-0d034ea1c133'...,[KRA382006844],NaN
2,cb50b866-5dc2-4b30-ac25-4293aedabcea,100,073803e3-e3cf-39ec-8a5d-fc4cf513a274,SWIM,152000,None,"[{'name': 'BTS', 'artist': {'id': '0d79fe8e-ba...",2026-05-01,[{'id': 'c8ed1c51-80ba-483f-8f39-37e283361981'...,[US23A8692629],part of “Today’s Hits: May 2026” DJ‐mix
3,a6b2a6a1-acdc-47ae-8022-03bfe864d534,100,073803e3-e3cf-39ec-8a5d-fc4cf513a274,JUMP,237000,None,"[{'name': 'BTS', 'artist': {'id': '0d79fe8e-ba...",2014-02-12,[{'id': 'e9535c7e-3388-472c-aca2-2bd468a8ba14'...,NaN,NaN
4,27fa7214-ce2c-4a61-9a65-3bd1a9a12301,100,073803e3-e3cf-39ec-8a5d-fc4cf513a274,Outro : Propose,122533,None,"[{'name': 'BTS', 'artist': {'id': '0d79fe8e-ba...",2014-02-12,[{'id': 'e9535c7e-3388-472c-aca2-2bd468a8ba14'...,NaN,NaN


In [14]:
recordings_summary = recordings_df[
    ["id", "title", "length", "first-release-date"]
].copy()

recordings_summary["duration_seconds"] = (
    recordings_summary["length"] / 1000
)

display(recordings_summary)

,id,title,length,first-release-date,duration_seconds
0,0ba51e80-b418-4402-9ab3-0cdd4a024873,A Brand New Day,205000,2019-06-14,205.000
1,5793a212-fa86-45a3-9b10-32edbb2db2bb,A Song Make to You Smile,189173,2020-12-10,189.173
2,cb50b866-5dc2-4b30-ac25-4293aedabcea,SWIM,152000,2026-05-01,152.000
3,a6b2a6a1-acdc-47ae-8022-03bfe864d534,JUMP,237000,2014-02-12,237.000
4,27fa7214-ce2c-4a61-9a65-3bd1a9a12301,Outro : Propose,122533,2014-02-12,122.533
5,8a5ace90-a9ef-41d0-ab65-041aacf1ee98,어디에서 왔는지 (inst.),240520,2014-05-14,240.520
6,93072fc7-a279-4f40-a0e2-664b9c26cb8d,상남자 (Boy in Luv),230400,2014-02-12,230.400
7,172ceb71-1a9a-4387-8fb4-2c5ad876debd,하루만 (inst.),239853,2014-05-14,239.853
8,ae5dd7e1-4575-46b3-8c6a-d3f70a03698b,Skit : Soulmate,92791,2014-02-12,92.791
9,511ee358-16ea-4b1e-85d4-3f7f9b56e59d,JUMP (inst.),237186,2014-05-14,237.186


In [15]:
print("Grabaciones:", len(recordings_summary))

print("IDs únicos:", recordings_summary["id"].nunique())

print("IDs duplicados:", recordings_summary["id"].duplicated().sum())

print("\nValores nulos:")
print(recordings_summary.isna().sum())

Grabaciones: 10
IDs únicos: 10
IDs duplicados: 0

Valores nulos:
id                    0
title                 0
length                0
first-release-date    0
duration_seconds      0
dtype: int64


In [16]:
print("Grabaciones devueltas:", len(recordings_data["recordings"]))
print("Total de resultados encontrados:", recordings_data.get("count"))
print("Límite de la consulta:", 10)

Grabaciones devueltas: 10
Total de resultados encontrados: 539
Límite de la consulta: 10


## MusicBrainz API | Resultados y conclusiones

### 1. Objetivo de la exploración

Evaluar MusicBrainz como posible fuente de metadatos musicales para el proyecto **Music Recommender | SQL Edition**.

Se ha analizado el acceso a la API, la estructura de las respuestas, la identificación de artistas y grabaciones, y su posible integración en una base de datos relacional.

### 2. Metodología

Se ha utilizado Python con las librerías `requests` y `pandas`.

Las consultas se han realizado mediante la API de MusicBrainz, solicitando respuestas en formato JSON.

Se ha seleccionado BTS como artista de prueba para evaluar los resultados.

### 3. Búsqueda de artistas

La primera consulta ha devuelto:

- Código HTTP: 200.
- Cinco resultados.
- Varios artistas con nombres iguales o similares.

El grupo BTS se ha identificado mediante:

- Nombre: BTS.
- Tipo: Group.
- País: KR.
- Descripción: South Korean boy group.
- Identificador MusicBrainz (MBID): `0d79fe8e-ba27-4859-bb8c-2f255f346853`.

**Hallazgo:** el nombre de un artista no garantiza su identificación única. Es preferible utilizar su MBID.

### 4. Consulta de grabaciones

Se ha utilizado el MBID de BTS para buscar grabaciones asociadas al artista.

La API ha devuelto información sobre:

- Identificador de grabación.
- Título.
- Duración en milisegundos.
- Fecha del primer lanzamiento.
- Artistas acreditados.
- Lanzamientos relacionados.
- Códigos ISRC.

Se ha creado un DataFrame con los campos principales y una columna calculada `duration_seconds`, convirtiendo los milisegundos a segundos.

### 5. Calidad de la muestra

Se han analizado diez grabaciones.

| Indicador | Resultado |
|---|---:|
| Grabaciones analizadas | 10 |
| Identificadores únicos | 10 |
| Identificadores duplicados | 0 |
| Valores nulos en los campos seleccionados | 0 |

La muestra presenta identificadores únicos y campos completos.

Estos resultados no permiten generalizar la calidad de todo el catálogo MusicBrainz.

### 6. Paginación

La consulta de grabaciones ha mostrado:

- Resultados encontrados: 539.
- Resultados devueltos: 10.
- Límite configurado: 10.

Esto demuestra que MusicBrainz permite recuperar resultados de manera limitada y que sería necesario utilizar paginación para consultar conjuntos más amplios.

### 7. Posible integración en SQL

MusicBrainz permite plantear entidades relacionadas como:

- `artists`: artistas identificados mediante MBID.
- `recordings`: grabaciones con identificadores propios.
- `releases`: lanzamientos musicales.
- Tablas intermedias para representar relaciones entre artistas, grabaciones y lanzamientos.

Los identificadores MusicBrainz pueden facilitar la vinculación de información procedente de distintas consultas.

El diseño definitivo deberá considerar que una grabación, una composición y un lanzamiento no representan necesariamente la misma entidad.

### 8. Ventajas y limitaciones

**Ventajas:**

- Acceso a información musical estructurada mediante API.
- Identificadores únicos para diferentes entidades.
- Información sobre artistas, grabaciones y lanzamientos.
- Posibilidad de consultar registros concretos sin descargar todo el catálogo.
- Potencial para enriquecer una base de datos SQL.

**Limitaciones:**

- Necesidad de realizar peticiones HTTP.
- Uso de paginación para recuperar grandes conjuntos de resultados.
- Posibles valores ausentes o inconsistencias en el catálogo general, todavía no evaluados.
- Las consultas realizadas no han proporcionado características acústicas como energía, bailabilidad o valencia.
- La muestra analizada es reducida.

### 9. Comparación con los datasets anteriores

| Fuente | Principal utilidad |
|---|---|
| Spotify Tracks 114K | Recomendaciones basadas en características musicales |
| FMA | Catálogo estructurado de canciones, artistas, álbumes y géneros |
| MusicBrainz | Metadatos adicionales e identificación de entidades musicales |

MusicBrainz podría complementar cualquiera de los dos datasets anteriores.

### 10. Conclusión

MusicBrainz es una fuente prometedora para enriquecer los metadatos del proyecto y trabajar relaciones entre entidades musicales.

No se considera, por el momento, un sustituto directo de los datasets con características de audio.

**Valoración provisional:** utilizar MusicBrainz como fuente complementaria de información musical.

La decisión definitiva se tomará después de evaluar Last.fm y seleccionar la arquitectura de datos del proyecto.

**Estado: exploración inicial de MusicBrainz completada.**